# Clean USDA SNAP State Monthly Data → MySQL
**Instructions**

1. Update Setup steps
2. Leave `DRY_RUN = True` to see printed summary before committing changes to the database.
3. Run final cell with `DRY_RUN = False` to commit changes to database.


**Process Notes**

For each fiscal year listed in `FISCAL_YEARS` (FY89 onward), this notebook:

1. Finds the workbook in `DATA_DIR` (`FY89.xls` … `FY26.xlsx`) and flattens every tab. It handles both USDA layouts: FY89–FY14, where Cost is the last column, and FY15 onward, where Cost is the third column. The layout is detected from each sheet's header.
2. Drops the regional summary rows, converts states to two-letter codes, and splits the data into `df_states` and `df_us`.
3. Checks each year: right fiscal year in the file, 12 months per state, and states adding up to the US Summary.
4. Loads `usda.snap_state_monthly` and `usda.snap_national_monthly`. It first checks that every value fits the table's column types, then deletes any rows already there for those fiscal years and appends the new rows. All of this runs in one transaction, so a failure leaves the tables unchanged.

**Data notes**

- Puerto Rico has received a block grant instead of SNAP since 1982, so it has no rows before FY20 and blank rows from FY20 on. American Samoa and the Northern Mariana Islands follow the same pattern.
- FY19: January 2019 is high and February 2019 is low because February benefits were issued early during the government shutdown. USDA footnotes these months as `/2`.
- A year in progress, such as FY26, has blank rows for months not yet reported. Rerunning that year once the file is updated replaces them.

In [48]:
# Before first run on new machine
#!pip install sqlalchemy 
#!pip install pymysql 
#!pip install xlrd

# Setup

In [49]:
#setup = 'Marie'
setup = 'Adam'
#setup = 'Casey'

In [50]:
# Folder holding the source workbooks, named FY25.xlsx, FY26.xlsx, ...
DATA_DIR = Path(
    r"G:\Shared drives\Center for Best Practices\Data Analytics\Data Admin\USDA\Data\snap_state_monthly\20261124"
)

In [60]:
# Set date range
# You must process at least two years at once (only one year will error out).
# Enter the first and last years you want to process below in "range(XXXX, XXXX)". For example, range(2025, 2026).
FISCAL_YEARS = [f"FY{y % 100:02d}" for y in range(2025, 2026)]

## Configuration

In [52]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import bindparam, create_engine, text, inspect

In [53]:
SAVE_CSV = True      # also write cleaned CSVs next to each workbook
LOAD_TO_DB = True    # set False to parse/validate without touching the database
DRY_RUN = True       # True = report what would be deleted/inserted, then roll back

# Database (MySQL on AWS RDS). Credentials are read from environment variables
# so they never live in the notebook. See the "Database" section for setup.
DB_HOST = os.environ.get("SNAP_DB_HOST", "your-instance.xxxxxxxx.us-east-1.rds.amazonaws.com")
DB_PORT = int(os.environ.get("SNAP_DB_PORT", "3306"))
DB_NAME = os.environ.get("SNAP_DB_NAME", "usda")
DB_USER = os.environ.get("SNAP_DB_USER")
DB_PASSWORD = os.environ.get("SNAP_DB_PASSWORD")
# RDS certificate bundle: https://truststore.pki.rds.amazonaws.com/global/global-bundle.pem
DB_SSL_CA = os.environ.get("SNAP_DB_SSL_CA", r"C:\certs\global-bundle.pem")

# Destination tables. Dataframe column names match the database columns, so
# these lists just pick which columns go to each table. mysql_primary_id and
# Run_Timestamp are filled in by MySQL.
DB_SCHEMA = "usda"
TABLE_STATES = "snap_state_monthly"
TABLE_US = "snap_national_monthly"
FY_DB_COLUMN = "Federal_Fiscal_Year"

STATE_DB_COLUMNS = [
    "State", "Region", "Month_Date", "Federal_Fiscal_Year",
    "Participation_Households", "Participation_Persons", "Cost",
    "Cost_Per_Household", "Cost_Per_Persons",
]
NATIONAL_DB_COLUMNS = [c for c in STATE_DB_COLUMNS if c not in ("State", "Region")]

# Regional summary blocks to drop, and the national block to keep separately
REGION_NAMES = {"MARO", "MPRO", "MWRO", "NERO", "SERO", "SWRO", "WRO"}
NATIONAL_NAME = "US Summary"

STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX",
    "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    # Territories
    "American Samoa": "AS", "Guam": "GU", "Puerto Rico": "PR",
    "Virgin Islands": "VI", "Commonwealth of Northern Mariana Islands": "MP",
}

NUM_COLS = [
    "Participation_Households",
    "Participation_Persons",
    "Cost",
    "Cost_Per_Household",
    "Cost_Per_Persons",
]

# Matches 'Oct 2024' and also footnoted months like 'Jan 2019 /2'
MONTH_PATTERN = re.compile(
    r"^((?:Oct|Nov|Dec|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep)\s+\d{4})\b"
)

# Order of the five number columns (B-F). USDA moved Cost from last to first
# in FY15, so the layout is detected from each sheet's header.
LAYOUT_CURRENT = ["Participation_Households", "Participation_Persons", "Cost",
                  "Cost_Per_Household", "Cost_Per_Persons"]  # FY15 onward
LAYOUT_OLD = ["Participation_Households", "Participation_Persons",
              "Cost_Per_Household", "Cost_Per_Persons", "Cost"]  # FY89-FY14

## Parsing

In [54]:
def detect_layout(df):
    """Return the column order for this sheet based on where 'Cost' sits in the header."""
    header = df.iloc[:10, 1:6].astype(str).apply(lambda c: c.str.strip())
    if (header.iloc[:, 4] == "Cost").any():   # column F says 'Cost'
        return LAYOUT_OLD
    if (header.iloc[:, 2] == "Cost").any():   # column D says 'Cost'
        return LAYOUT_CURRENT
    raise ValueError("Unrecognized header layout: could not find the 'Cost' column.")


def parse_snap_sheet(df):
    """Parse one raw (header=None) sheet into a flat table of monthly rows.

    A row is treated as a location title when it has text in column A and
    nothing in the numeric columns (e.g. 'Connecticut', 'NERO'). Every month
    row that follows is assigned to that location until the next title.
    'Total' rows are skipped on purpose (they can be recomputed).
    """
    records = []
    current_location = None
    columns = detect_layout(df)

    for _, row in df.iterrows():
        first = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""
        numbers_empty = row.iloc[1:6].isna().all()
        month = MONTH_PATTERN.match(first)

        if month:
            if current_location is None:
                raise ValueError(f"Found month row '{first}' before any location title.")
            records.append(
                {
                    "State": current_location,
                    "Month": month.group(1),  # drops footnote markers like ' /2'
                    **dict(zip(columns, row.iloc[1:6].tolist())),
                }
            )
        elif first.startswith("Footnotes"):
            break  # nothing useful below this point
        elif first and numbers_empty and first != "Total":
            # Skip the report header lines at the top of each sheet
            if first.startswith(("National Data Bank", "SNAP Monthly", "Fiscal Year", "No data found")):
                continue
            current_location = first

    return pd.DataFrame(records)


def load_workbook(file_path):
    """Read every tab, parse it, and return one combined, typed DataFrame."""
    excel_file = pd.ExcelFile(file_path)
    frames = []

    for sheet_name in excel_file.sheet_names:
        df_raw = pd.read_excel(excel_file, sheet_name=sheet_name, header=None)
        df_parsed = parse_snap_sheet(df_raw)
        if df_parsed.empty:
            print(f"    Skipped tab '{sheet_name}' (no monthly rows found).")
            continue
        df_parsed["Region"] = sheet_name.strip()  # older files have 'SWRO ' etc.
        frames.append(df_parsed)

    if not frames:
        raise ValueError(f"No monthly data blocks were found in {file_path}.")

    df = pd.concat(frames, ignore_index=True)

    # Clean numeric columns: strip '$' / ',' and treat '--' as missing
    for col in NUM_COLS:
        cleaned = df[col].astype(str).str.replace(r"[\$,]", "", regex=True).str.strip()
        df[col] = pd.to_numeric(cleaned.replace({"--": np.nan, "nan": np.nan}), errors="coerce")

    # Real date column (first of the month) so the data sorts and plots correctly
    df["Month_Date"] = pd.to_datetime(df["Month"], format="%b %Y")
    # Federal fiscal year: Oct-Dec belong to the next calendar year's FY
    df["Federal_Fiscal_Year"] = df["Month_Date"].dt.year + (df["Month_Date"].dt.month >= 10)

    return df

## Split into state-level and national tables

In [55]:
def split_tables(df):
    """Return (df_states, df_us) with regional summary rows removed."""
    df = df[~df["State"].isin(REGION_NAMES)]

    df_us = df[df["State"] == NATIONAL_NAME].copy()
    df_states = df[df["State"] != NATIONAL_NAME].copy()

    # Replace full names with two-letter codes; fail loudly on anything unknown
    unknown = sorted(set(df_states["State"]) - set(STATE_ABBREV))
    if unknown:
        raise ValueError(f"No abbreviation defined for: {unknown}. Add them to STATE_ABBREV.")
    df_states["State"] = df_states["State"].map(STATE_ABBREV)
    df_us["State"] = "US"

    cols = ["State", "Region", "Federal_Fiscal_Year", "Month", "Month_Date", *NUM_COLS]
    df_states = df_states[cols].sort_values(["State", "Month_Date"]).reset_index(drop=True)
    df_us = df_us[[c for c in cols if c != "Region"]].sort_values("Month_Date").reset_index(drop=True)

    return df_states, df_us


def validate(df_states, df_us, expected_fy):
    """Sanity checks: right fiscal year, 12 months per state, states sum to US."""
    found_fys = set(df_states["Federal_Fiscal_Year"]) | set(df_us["Federal_Fiscal_Year"])
    if found_fys != {expected_fy}:
        raise ValueError(f"Expected only FY{expected_fy} data but found fiscal years {sorted(found_fys)}.")

    months_per_state = df_states.groupby("State")["Month"].nunique()
    short = months_per_state[months_per_state != 12]
    if not short.empty:
        print(f"    WARNING: states without 12 months of data:\n{short.to_string()}")

    missing = df_states[df_states[NUM_COLS].isna().any(axis=1)]
    if not missing.empty:
        print(f"    Note: rows with blank values by state: {missing['State'].value_counts().to_dict()}")

    check_cols = ["Participation_Households", "Participation_Persons", "Cost"]
    state_sums = df_states.groupby("Month_Date")[check_cols].sum()
    us = df_us.set_index("Month_Date")[check_cols]
    diff = (state_sums - us).abs().max()
    if (diff > 0).any():
        print(f"    WARNING: sum of states differs from US Summary (max monthly gap):\n{diff.to_string()}")
    else:
        print("    Sum of states matches US Summary for every month.")

## Process multiple fiscal years

In [56]:
def find_workbook(data_dir, fy_label):
    """Locate the workbook for a label like 'FY25' (FY25.xlsx, FY25_20260601.xlsx, ...)."""
    matches = sorted(
        p for p in Path(data_dir).glob(f"{fy_label}*.xls*") if not p.name.startswith("~$")
    )
    if not matches:
        raise FileNotFoundError(f"No workbook starting with '{fy_label}' in {data_dir}")
    if len(matches) > 1:
        names = ", ".join(p.name for p in matches)
        raise FileExistsError(f"More than one workbook for {fy_label}: {names}. Keep only one.")
    return matches[0]


def fy_label_to_year(fy_label):
    """'FY25' -> 2025, 'FY89' -> 1989 (two-digit years 89-99 are 1900s)."""
    m = re.fullmatch(r"FY(\d{2})", fy_label.strip().upper())
    if not m:
        raise ValueError(f"Fiscal year labels should look like 'FY25', got '{fy_label}'.")
    yy = int(m.group(1))
    return (1900 if yy >= 89 else 2000) + yy


def process_fiscal_years(data_dir, fy_labels, save_csv=True):
    """Parse and validate every requested year. Returns (df_states, df_us) for all years."""
    all_states, all_us = [], []

    for label in fy_labels:
        fy = fy_label_to_year(label)
        path = find_workbook(data_dir, label)
        print(f"\n{label}: reading {path.name}")

        df_states, df_us = split_tables(load_workbook(path))
        validate(df_states, df_us, fy)
        print(f"    {len(df_states)} state rows, {len(df_us)} US rows")

        if save_csv:
            df_states.to_csv(path.with_name(f"{path.stem}_states.csv"), index=False)
            df_us.to_csv(path.with_name(f"{path.stem}_us_summary.csv"), index=False)

        all_states.append(df_states)
        all_us.append(df_us)

    return pd.concat(all_states, ignore_index=True), pd.concat(all_us, ignore_index=True)

## Database

In [57]:
## Database Connection to mysql 8.0

if setup == 'Marie':
    directory = 'G:\Shared drives\Center for Best Practices\Data Analytics\Data Admin\Database administration\Connections'
    sys.path.insert(1, directory)
    #Import AWS Data Lake connection settings
    import aws_data_lake_connection_mabbondanza_v8 as aws

elif setup == 'Casey':
    directory = 'G:\Shared drives\Center for Best Practices\Data Analytics\Data Admin\Database administration\Connections'
    sys.path.insert(1, directory)
    #Import AWS Data Lake connection settings
    import aws_data_lake_connection_awozniak_v8 as aws 

elif setup == 'Adam':
    directory = 'G:\Shared drives\Center for Best Practices\Data Analytics\Data Admin\Database administration\Connections'
    sys.path.insert(1, directory)
    #Import AWS Data Lake connection settings
    import aws_data_lake_connection_awozniak_v8 as aws 

engine_usda = create_engine('mysql+pymysql://'+aws.info['username']+':'+aws.info['password']+'@'+aws.info['rds_host']+'/usda')

conn_usda = engine_usda.connect()

#print (engine.table_names())

insp = inspect(engine_usda)
print(insp.get_table_names())

['child_nutrition_TO_DELETE', 'national_enrollment_total', 'national_nslp_sbp', 'snap_national_monthly', 'snap_state_monthly', 'state_201909_to_201912', 'state_enrollment', 'state_fiscal_year', 'state_monthly']


In [58]:
# One-time setup:
#   pip install sqlalchemy pymysql xlrd   (xlrd reads the older .xls files)
#   Set environment variables SNAP_DB_HOST, SNAP_DB_NAME, SNAP_DB_USER,
#   SNAP_DB_PASSWORD (and optionally SNAP_DB_PORT, SNAP_DB_SSL_CA).
#   On Windows: setx SNAP_DB_USER "myuser"  (then restart Jupyter)
#   Download the RDS CA bundle and point SNAP_DB_SSL_CA at it.
# def get_engine():
#     if not DB_USER or not DB_PASSWORD:
#         raise RuntimeError("Set SNAP_DB_USER and SNAP_DB_PASSWORD environment variables first.")
#     url = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
#     # Hide the password if the URL ever gets printed
#     return create_engine(
#         url,
#         connect_args={"ssl": {"ca": DB_SSL_CA}} if DB_SSL_CA else {},
#         pool_pre_ping=True,
#         hide_parameters=True,
#     )


INT_RANGES = {  # signed ranges for MySQL integer column types
    "tinyint": 2**7 - 1, "smallint": 2**15 - 1, "mediumint": 2**23 - 1,
    "int": 2**31 - 1, "bigint": 2**63 - 1,
}


def to_db_frame(df, columns):
    """Keep the table's columns and convert types so MySQL stores them exactly."""
    out = df[columns].copy()
    out["Month_Date"] = out["Month_Date"].dt.date
    for col in ["Participation_Households", "Participation_Persons", "Cost"]:
        out[col] = out[col].round().astype("Int64")  # whole numbers, NULL-safe
    for col in ["Cost_Per_Household", "Cost_Per_Persons"]:
        out[col] = out[col].round(2)  # match decimal(10,2)
    return out.astype(object).where(out.notna(), None)  # NaN/NA -> NULL


def check_fits_schema(conn, table, df):
    """Stop before writing anything if a column is missing or a value won't fit.

    Reads the live column definitions, so after an ALTER TABLE (e.g. INT ->
    BIGINT) the check picks up the new type automatically.
    """
    rows = conn.execute(
        text(
            "SELECT COLUMN_NAME, DATA_TYPE, NUMERIC_PRECISION, NUMERIC_SCALE, "
            "CHARACTER_MAXIMUM_LENGTH FROM information_schema.COLUMNS "
            "WHERE TABLE_SCHEMA = :s AND TABLE_NAME = :t"
        ),
        {"s": DB_SCHEMA, "t": table},
    ).fetchall()
    schema = {r[0]: r for r in rows}
    if not schema:
        raise RuntimeError(f"Table {DB_SCHEMA}.{table} not found.")

    problems = []
    for col in df.columns:
        if col not in schema:
            problems.append(f"{col}: column not in table")
            continue
        _, dtype, precision, scale, max_len = schema[col]
        values = df[col].dropna()
        if values.empty:
            continue
        if dtype in INT_RANGES:
            biggest = max(abs(int(v)) for v in values)
            if biggest > INT_RANGES[dtype]:
                problems.append(
                    f"{col}: largest value {biggest:,} exceeds {dtype.upper()} limit "
                    f"{INT_RANGES[dtype]:,} (change the column to BIGINT)"
                )
        elif dtype == "decimal":
            limit = 10 ** (int(precision) - int(scale))
            biggest = max(abs(float(v)) for v in values)
            if biggest >= limit:
                problems.append(f"{col}: largest value {biggest:,} too large for DECIMAL({precision},{scale})")
        elif dtype in ("varchar", "char") and max_len:
            longest = max(len(str(v)) for v in values)
            if longest > max_len:
                problems.append(f"{col}: value length {longest} exceeds {dtype}({max_len})")

    if problems:
        raise ValueError(f"Data does not fit {DB_SCHEMA}.{table}:\n  " + "\n  ".join(problems))


def existing_row_counts(conn, table, fiscal_years):
    """Return {fiscal_year: row_count} already in the table for these years."""
    stmt = text(
        f"SELECT {FY_DB_COLUMN} AS fy, COUNT(*) AS n FROM {DB_SCHEMA}.{table} "
        f"WHERE {FY_DB_COLUMN} IN :fys GROUP BY {FY_DB_COLUMN}"
    ).bindparams(bindparam("fys", expanding=True))
    return {int(r.fy): int(r.n) for r in conn.execute(stmt, {"fys": fiscal_years})}


def replace_fiscal_years(engine, df_states, df_us, dry_run=True):
    """Delete existing rows for the fiscal years being loaded, then append the new rows.

    Everything runs in one transaction: if any insert fails, the deletes are
    rolled back and the tables are left exactly as they were. With dry_run=True
    the work is done and reported, then rolled back.
    """
    fiscal_years = sorted(
        int(y) for y in set(df_states[FY_DB_COLUMN]) | set(df_us[FY_DB_COLUMN])
    )
    loads = [
        (TABLE_STATES, to_db_frame(df_states, STATE_DB_COLUMNS)),
        (TABLE_US, to_db_frame(df_us, NATIONAL_DB_COLUMNS)),
    ]

    conn = engine.connect()
    trans = conn.begin()
    try:
        # Check both tables before deleting anything
        for table, df in loads:
            check_fits_schema(conn, table, df)

        for table, df in loads:
            before = existing_row_counts(conn, table, fiscal_years)
            if before:
                print(f"  {table}: deleting existing rows {before}")
                delete = text(
                    f"DELETE FROM {DB_SCHEMA}.{table} WHERE {FY_DB_COLUMN} IN :fys"
                ).bindparams(bindparam("fys", expanding=True))
                conn.execute(delete, {"fys": fiscal_years})
            else:
                print(f"  {table}: no existing rows for FY {fiscal_years}")

            df.to_sql(
                table, con=conn, schema=DB_SCHEMA, if_exists="append",
                index=False, method="multi", chunksize=500,
            )
            after = existing_row_counts(conn, table, fiscal_years)
            print(f"  {table}: inserted {len(df)} rows -> now {after}")

        if dry_run:
            trans.rollback()
            print("\nDRY RUN: all changes rolled back. Set DRY_RUN = False to commit.")
        else:
            trans.commit()
            print("\nCommitted.")
    except Exception:
        trans.rollback()
        print("\nError: all changes rolled back.")
        raise
    finally:
        conn.close()

## Run

In [61]:
df_states, df_us = process_fiscal_years(DATA_DIR, FISCAL_YEARS, save_csv=False)
print(f"\nTotal: {len(df_states)} state rows, {len(df_us)} US rows")

if LOAD_TO_DB:
    #engine = get_engine()
    engine = engine_usda
    replace_fiscal_years(engine, df_states, df_us, dry_run=False)


FY25: reading FY25.xlsx
    Note: rows with blank values by state: {'AS': 12, 'MP': 12, 'PR': 12}
    Sum of states matches US Summary for every month.
    672 state rows, 12 US rows

Total: 672 state rows, 12 US rows
  snap_state_monthly: deleting existing rows {2025: 672}
  snap_state_monthly: inserted 672 rows -> now {2025: 672}
  snap_national_monthly: deleting existing rows {2025: 12}
  snap_national_monthly: inserted 12 rows -> now {2025: 12}

Committed.
